In [1]:
import numpy as np
import yfinance as yf
from scipy.stats import norm
from datetime import datetime

In [2]:
asset = yf.download("AMZN", period="5d")["Close"]["AMZN"]
S = asset.iloc[-1]

print(f"S : $ {S:.2f}")

[*********************100%***********************]  1 of 1 completed

S : $ 212.79


In [3]:
asset = yf.Ticker("AMZN")
date = asset.options[13]
expiry = datetime.strptime(date, "%Y-%m-%d")
today = datetime.today()

T = (expiry - today).days / 365

print(f"T : {T:.4f} years")

T : 0.3699 years


In [4]:
chain = asset.option_chain(date)

puts = chain.puts

K = puts["strike"]
sigma = puts["impliedVolatility"]

lookup = puts[puts["strike"] >= S][["strike", "impliedVolatility"]]
print(lookup.head(30))

    strike  impliedVolatility
21   215.0           0.000010
22   220.0           0.000010
23   225.0           0.000010
24   230.0           0.000010
25   235.0           0.000010
26   240.0           0.000010
27   245.0           0.000010
28   250.0           0.000010
29   255.0           0.000010
30   260.0           0.000010
31   265.0           0.000010
32   270.0           0.000010
33   275.0           0.000010
34   280.0           0.000010
35   285.0           0.000010
36   290.0           0.000010
37   295.0           0.000010
38   300.0           0.000010
39   305.0           0.000010
40   310.0           0.532597
41   315.0           0.546330
42   320.0           0.561345
43   325.0           0.000010
44   335.0           0.000010
45   340.0           0.000010
46   350.0           0.638126
47   370.0           0.685855


In [5]:
K = K.iloc[41]
sigma = sigma.iloc[41]

print(f"K       : $ {K:.2f}")
print(f"sigma   : {sigma:.4f}")

K       : $ 315.00
sigma   : 0.5463


In [6]:
r = yf.download("^TNX", period = "5d")["Close"]["^TNX"]
r = r.iloc[-1]
r = r / 100

print(f"r : {r:.4f}")

[*********************100%***********************]  1 of 1 completed

r : 0.0433


In [7]:
d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

Nd1 =  norm.cdf(-d1)
Nd2 =  norm.cdf(-d2)

print(f"N(d1) : {Nd1:.4f}")
print(f"N(d2) : {Nd2:.4f}")

N(d1) : 0.8330
N(d2) : 0.9029


In [ ]:
put_price = K * np.exp(-r * T) * Nd2 - S * Nd1
print(f"Put Price : $ {put_price:.2f}")

Put Price : $ 102.64


In [17]:
current_put_price = puts["lastPrice"]
current_put_price = current_put_price.iloc[41]

if current_put_price > put_price:
    print("Overvalued")
elif current_put_price < put_price:
    print("Undervalued")
else:
    print("Fair Value")

compare_price = current_put_price - put_price
print(f"Put Price BSM       : $ {put_price:.2f}")
print(f"Put Price Current   : $ {current_put_price:.2f}")
print(f"Mispricing Gap      : $ {compare_price:.2f}")

Undervalued
Put Price BSM       : $ 102.64
Put Price Current   : $ 77.25
Mispricing Gap      : $ -25.39
